# Módulo 02 · Aula 02 — Branches e Remotos

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

Na aula anterior você trabalhou numa linha do tempo só. Funciona — até você precisar mexer em duas coisas ao mesmo tempo, ou trabalhar com outra pessoa.

> *"Estou no meio da refatoração do relatório e a diretora precisa de uma correção urgente em produção. Se eu commitar agora, subo código pela metade. Se não commitar, perco o trabalho."*

Branches resolvem exatamente isso.

## O que você vai aprender aqui

| # | Tópico | Por que importa |
|---|--------|-----------------|
| 1 | O que é um branch (de verdade) | É mais simples do que parece |
| 2 | `HEAD` e `switch` | Navegar entre linhas do tempo |
| 3 | Merge fast-forward vs 3-way | Os dois tipos de junção |
| 4 | **Conflitos** | Resolver sem pânico |
| 5 | Estratégias de branch | Como times reais organizam |
| 6 | Remotos e GitHub | Colaboração |
| 7 | `clone`, `push`, `pull`, `fetch` | O ciclo com o servidor |
| 8 | SSH vs HTTPS | Autenticação |
| 9 | Pull Requests | O fluxo profissional |

## ⚙️ Preparando o laboratório

Como na aula anterior, tudo acontece numa pasta descartável.

In [ ]:
import shutil
import subprocess
from pathlib import Path

BASE = Path("lab_git_02").resolve()
shutil.rmtree(BASE, ignore_errors=True)
BASE.mkdir(parents=True)

REPO = BASE / "atlas"
REPO.mkdir()


def git(*args, cwd=REPO, mostrar=True):
    """Executa git e imprime a saída."""
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if mostrar:
        print("$ git " + " ".join(args))
        saida = (r.stdout + r.stderr).rstrip()
        print(saida if saida else "(sem saída)")
        print()
    return r


def escrever(nome, conteudo, pasta=REPO):
    caminho = Path(pasta) / nome
    caminho.parent.mkdir(parents=True, exist_ok=True)
    caminho.write_text(conteudo, encoding="utf-8")


def ler(nome, pasta=REPO):
    return (Path(pasta) / nome).read_text(encoding="utf-8")


# Repositório base com alguns commits
git("init", "-b", "main")
git("config", "user.name", "Aluno Atlas")
git("config", "user.email", "aluno@aurora.com.br")

escrever("README.md", "# Atlas\n\nSistema de relatórios da Aurora Comércio.\n")
git("add", "."); git("commit", "-m", "chore: estrutura inicial", mostrar=False)

escrever("metricas.py", '''"""Métricas de vendas."""

STATUS_FATURAVEL = "pago"


def faturamento(vendas):
    """Soma o valor dos pedidos faturáveis."""
    return sum(v["qtd"] * v["preco"] for v in vendas if v["status"] == STATUS_FATURAVEL)
''')
git("add", "."); git("commit", "-m", "feat: adiciona cálculo de faturamento", mostrar=False)

git("log", "--oneline")
print("📁 Laboratório:", BASE)

## 1. O que é um branch

Aqui está o segredo mais bem guardado do Git:

> **Um branch é apenas um arquivo de 41 bytes contendo o hash de um commit.**

É isso. Não é uma cópia da pasta, não é um diretório paralelo. É um **ponteiro móvel**.

```
       ┌──────┐     ┌──────┐     ┌──────┐
       │  A   │◀────│  B   │◀────│  C   │
       └──────┘     └──────┘     └──────┘
                                     ▲
                                     │
                                   main        ← o ponteiro
                                     ▲
                                     │
                                   HEAD        ← onde VOCÊ está
```

Cada commit aponta para o **pai** (a seta vai para trás — o Git sempre olha para o passado). O branch aponta para o commit mais recente daquela linha. `HEAD` aponta para o branch atual.

Quando você faz um commit novo, o branch **avança** automaticamente:

```
       ┌──────┐     ┌──────┐     ┌──────┐     ┌──────┐
       │  A   │◀────│  B   │◀────│  C   │◀────│  D   │
       └──────┘     └──────┘     └──────┘     └──────┘
                                                  ▲
                                                main ← avançou
```

**Por isso criar um branch no Git é instantâneo** — é escrever 41 bytes em um arquivo. Em sistemas antigos, criar um branch copiava o projeto inteiro e levava minutos.

In [ ]:
# Prova: o branch é literalmente um arquivo
ref = REPO / ".git" / "refs" / "heads" / "main"
print("Arquivo:", ref)
print("Conteúdo:", ref.read_text().strip())
print("Tamanho:", ref.stat().st_size, "bytes")
print()
print("HEAD aponta para:", (REPO / ".git" / "HEAD").read_text().strip())

## 2. Criando e navegando entre branches

| Comando | O que faz |
|---------|-----------|
| `git branch` | Lista os branches locais |
| `git branch -a` | Lista todos, inclusive remotos |
| `git branch nome` | Cria (mas **não** troca) |
| `git switch nome` | Troca para o branch |
| `git switch -c nome` | **Cria e troca** ← o mais usado |
| `git branch -d nome` | Apaga (só se já foi mesclado) |
| `git branch -D nome` | Apaga à força |
| `git branch -m novo` | Renomeia o branch atual |

> 💡 **`switch` vs `checkout`.** Você vai ver muito `git checkout -b` em tutoriais antigos. O `checkout` faz coisas demais (troca branch, restaura arquivo, navega para commit) e por isso é confuso. Desde o Git 2.23 existem dois comandos separados e mais claros:
>
> - **`git switch`** → trocar de branch
> - **`git restore`** → restaurar arquivos
>
> Use os novos. O `checkout` continua funcionando e você vai encontrá-lo em código alheio, mas não precisa aprendê-lo primeiro.

In [ ]:
git("branch")                      # só existe main
git("switch", "-c", "feature/relatorio-por-canal")
git("branch")

In [ ]:
# Trabalhando no branch novo
escrever("metricas.py", '''"""Métricas de vendas."""

from collections import defaultdict

STATUS_FATURAVEL = "pago"


def faturamento(vendas):
    """Soma o valor dos pedidos faturáveis."""
    return sum(v["qtd"] * v["preco"] for v in vendas if v["status"] == STATUS_FATURAVEL)


def por_canal(vendas):
    """Agrupa o faturamento por canal de venda."""
    total = defaultdict(float)
    for v in vendas:
        if v["status"] == STATUS_FATURAVEL:
            total[v["canal"]] += v["qtd"] * v["preco"]
    return dict(total)
''')

git("add", ".")
git("commit", "-m", "feat(metricas): adiciona agrupamento por canal")
git("log", "--oneline", "--all", "--graph")

```
* c3d4e5f (HEAD -> feature/relatorio-por-canal) feat(metricas): adiciona agrupamento por canal
* b2c3d4e (main) feat: adiciona cálculo de faturamento
* a1b2c3d chore: estrutura inicial
```

Repare: **`main` ficou parado**. Ele ainda aponta para o commit anterior. As duas linhas do tempo existem simultaneamente.

In [ ]:
# Voltando para main — o arquivo volta ao estado antigo!
git("switch", "main")
print(">>> Conteúdo de metricas.py em main:")
print(ler("metricas.py"))
print(">>> A função por_canal existe aqui?", "por_canal" in ler("metricas.py"))

In [ ]:
git("switch", "feature/relatorio-por-canal")
print(">>> E aqui?", "por_canal" in ler("metricas.py"))

> 📌 **É o mesmo diretório.** O Git troca o conteúdo dos arquivos no disco quando você muda de branch. Não existem duas pastas — existe uma pasta que muda de conteúdo conforme o branch ativo.
>
> ⚠️ Por isso: **commite ou guarde suas mudanças antes de trocar de branch.** Se houver alterações não commitadas que conflitem, o Git recusa a troca. (A aula 03 ensina `git stash` para esse caso.)

## 3. Merge — juntando linhas do tempo

### Caso 1: fast-forward

Quando o branch de destino **não avançou** desde que você se separou dele, o Git só precisa mover o ponteiro para frente. Não há o que decidir.

```
ANTES:
       A ◀── B ◀── C          ← feature
             ▲
            main

DEPOIS de `git switch main; git merge feature`:
       A ◀── B ◀── C
                   ▲
              main, feature    ← main simplesmente "andou"
```

Nenhum commit novo é criado. É a junção mais limpa possível.

In [ ]:
git("switch", "main")
git("merge", "feature/relatorio-por-canal")
git("log", "--oneline", "--graph", "--all")

In [ ]:
# Branch mesclado pode ser apagado com segurança (-d verifica isso)
git("branch", "-d", "feature/relatorio-por-canal")
git("branch")

### Caso 2: merge de 3 vias (*three-way merge*)

Quando **as duas linhas avançaram**, não dá para só mover o ponteiro. O Git compara três pontos:

1. O **ancestral comum** (onde vocês se separaram)
2. A ponta do branch atual
3. A ponta do branch que está sendo mesclado

E cria um **commit de merge** com **dois pais**.

```
ANTES:
                 D ◀── E        ← feature
                /
       A ◀── B ◀── C            ← main
             ▲
        ancestral comum

DEPOIS:
                 D ◀── E
                /        \
       A ◀── B ◀── C ◀──── M    ← main (M tem dois pais: C e E)
```

In [ ]:
# Duas frentes de trabalho ao mesmo tempo
git("switch", "-c", "feature/curva-abc")
escrever("curva_abc.py", '''"""Classificação ABC de praças."""


def classificar(faturamento_por_cidade):
    """Classifica cidades em A (80%), B (95%) e C (resto)."""
    total = sum(faturamento_por_cidade.values())
    ordenado = sorted(faturamento_por_cidade.items(), key=lambda kv: -kv[1])

    resultado, acumulado = [], 0.0
    for cidade, valor in ordenado:
        acumulado += valor / total if total else 0
        classe = "A" if acumulado <= 0.80 else "B" if acumulado <= 0.95 else "C"
        resultado.append({"cidade": cidade, "valor": valor, "classe": classe})
    return resultado
''')
git("add", "."); git("commit", "-m", "feat: adiciona curva ABC de praças", mostrar=False)

# Enquanto isso, main também avançou (outra pessoa, ou você mesmo)
git("switch", "main")
escrever("README.md", """# Atlas

Sistema de relatórios da Aurora Comércio.

## Instalação

```bash
python -m venv .venv
source .venv/bin/activate
```
""")
git("add", "."); git("commit", "-m", "docs: adiciona instruções de instalação", mostrar=False)

git("log", "--oneline", "--graph", "--all")

In [ ]:
# Agora o merge NÃO é fast-forward
git("merge", "feature/curva-abc", "-m", "merge: integra curva ABC de praças")
git("log", "--oneline", "--graph", "--all")

In [ ]:
# O commit de merge tem DOIS pais
git("log", "-1", "--pretty=format:hash=%h%nparents=%p%nsubject=%s")
git("branch", "-d", "feature/curva-abc")

### `--no-ff`: forçando o commit de merge

Alguns times preferem que **toda** integração de feature gere um commit de merge, mesmo quando o fast-forward seria possível. Motivo: o histórico registra visualmente "aqui entrou a feature X", e reverter a feature inteira vira um comando só.

```bash
git merge --no-ff feature/algo
```

| Estratégia | Histórico | Quando usar |
|------------|-----------|-------------|
| `--ff` (padrão) | Linear, limpo | Branches curtos, mudanças pequenas |
| `--no-ff` | Mostra os "balões" de cada feature | Times que querem rastreabilidade |
| `--squash` | Junta tudo num commit só | Feature com 40 commits de "wip" |

## 4. Conflitos

**Um conflito acontece quando duas linhas do tempo alteram as mesmas linhas do mesmo arquivo.**

O Git é bom em mesclar automaticamente. Ele resolve sozinho quando as mudanças estão em partes diferentes do arquivo. Ele só te chama quando **não tem como decidir** — e isso é correto: ele não sabe qual versão está certa. Você sabe.

> 🧘 **Conflito não é erro.** É o Git pedindo uma decisão. Ninguém quebrou nada.

In [ ]:
# Cenário: duas pessoas mudam a MESMA linha
escrever("config.py", '''"""Configuração do Atlas."""

TAXA_IMPOSTO = 0.18
FRETE_GRATIS_ACIMA_DE = 500.0
MOEDA = "BRL"
''')
git("add", "."); git("commit", "-m", "feat: adiciona arquivo de configuração", mostrar=False)

# Pessoa A, no branch fiscal
git("switch", "-c", "ajuste/fiscal", mostrar=False)
escrever("config.py", '''"""Configuração do Atlas."""

TAXA_IMPOSTO = 0.20
FRETE_GRATIS_ACIMA_DE = 500.0
MOEDA = "BRL"
''')
git("add", "."); git("commit", "-m", "fix(fiscal): atualiza alíquota para 20% conforme nova legislação", mostrar=False)

# Pessoa B, em main
git("switch", "main", mostrar=False)
escrever("config.py", '''"""Configuração do Atlas."""

TAXA_IMPOSTO = 0.175
FRETE_GRATIS_ACIMA_DE = 300.0
MOEDA = "BRL"
''')
git("add", "."); git("commit", "-m", "feat(comercial): reduz limite de frete grátis para R$ 300", mostrar=False)

git("log", "--oneline", "--graph", "--all")

In [ ]:
# O merge vai conflitar
git("merge", "ajuste/fiscal")

In [ ]:
# Como o Git deixou o arquivo:
print(ler("config.py"))

### Anatomia dos marcadores de conflito

```
<<<<<<< HEAD
TAXA_IMPOSTO = 0.175          ← a versão do branch em que VOCÊ está
=======
TAXA_IMPOSTO = 0.20           ← a versão que está ENTRANDO
>>>>>>> ajuste/fiscal
```

| Marcador | Significado |
|----------|-------------|
| `<<<<<<< HEAD` | Início da sua versão |
| `=======` | Divisor |
| `>>>>>>> branch` | Fim da versão que entra |

**Resolver = editar o arquivo até ficar como deve ficar, apagando os marcadores.**

Suas opções:

1. Ficar com a sua versão
2. Ficar com a versão que entra
3. **Combinar as duas** ← muitas vezes é a resposta certa
4. Escrever algo totalmente novo

Depois: `git add <arquivo>` (isso sinaliza "resolvido") e `git commit`.

In [ ]:
git("status")

In [ ]:
# Resolvendo: neste caso as duas mudanças são legítimas e independentes.
# A alíquota nova é lei; o limite de frete é decisão comercial. Ficamos com AS DUAS.
escrever("config.py", '''"""Configuração do Atlas."""

TAXA_IMPOSTO = 0.20            # nova legislação (ajuste/fiscal)
FRETE_GRATIS_ACIMA_DE = 300.0  # decisão comercial
MOEDA = "BRL"
''')

git("add", "config.py")
git("status")

In [ ]:
git("commit", "-m", "merge: integra alíquota fiscal de 20% com novo limite de frete")
git("log", "--oneline", "--graph", "--all")
git("branch", "-d", "ajuste/fiscal")

### Ferramentas para conflitos

| Comando | Para quê |
|---------|----------|
| `git merge --abort` | 🆘 **Desiste do merge e volta ao estado anterior.** Seu botão de pânico. |
| `git diff` | Durante o conflito, mostra as diferenças combinadas |
| `git checkout --ours arquivo` | Aceita a versão do branch atual, inteira |
| `git checkout --theirs arquivo` | Aceita a versão que está entrando, inteira |
| `git mergetool` | Abre a ferramenta visual configurada |

O **VS Code** detecta conflitos automaticamente e mostra botões acima do bloco: *Accept Current Change*, *Accept Incoming Change*, *Accept Both Changes*, *Compare Changes*. Para conflitos simples, é o caminho mais rápido.

> ⚠️ Cuidado com `--ours` e `--theirs`: eles descartam a outra versão **inteira**, não só a linha conflitante. Use apenas quando tiver certeza.

### Como ter menos conflitos

1. **Branches curtos.** Um branch que vive 2 dias conflita pouco; um que vive 3 semanas conflita muito.
2. **Sincronize com frequência.** `git pull` no `main` e mescle no seu branch regularmente, em vez de esperar o fim.
3. **Commits pequenos e focados.**
4. **Combine com o time** quem mexe em quê.
5. **Formatação consistente.** Metade dos conflitos reais é briga de formatador. Use `ruff`/`black` com a mesma configuração para todos.

## 5. Estratégias de branching

### Padrão de nomes

```
main                         # sempre estável, pronto para produção
develop                      # integração (só em times maiores)
feature/relatorio-por-canal  # nova funcionalidade
fix/calculo-frete            # correção
hotfix/erro-producao         # urgência em produção
chore/atualiza-dependencias  # manutenção
docs/readme-instalacao       # documentação
```

Use **kebab-case**, seja descritivo, e inclua o número do ticket se houver: `feature/AURORA-42-relatorio-canal`.

### GitHub Flow — o que você vai usar

O mais simples e o mais comum em times pequenos e médios:

```
main ────●────────●────────●────────●───▶  (sempre deployável)
          \      /          \      /
           ●──●─┘            ●──●─┘
        feature/A         feature/B
```

1. `main` está **sempre** funcionando
2. Toda mudança nasce em um branch a partir de `main`
3. Ao terminar, abre um **Pull Request**
4. Alguém revisa
5. CI roda os testes (Módulo 09)
6. Merge em `main` → deploy

**Regra número 1: nunca commite direto em `main`.** Nem quando "é só uma linha". Especialmente quando "é só uma linha".

### Git Flow — para quem tem versões

Mais elaborado, com `develop`, `release/*` e `hotfix/*`. Faz sentido para software com versões formais (v1.2, v1.3) e janelas de release. Para uma aplicação web com deploy contínuo, costuma ser excesso de burocracia.

## 6. Remotos — o repositório no servidor

Um **remoto** é um apelido para a URL de outro repositório. Por convenção, o principal se chama `origin`.

```
   Seu computador                      GitHub
 ┌────────────────┐                ┌────────────────┐
 │ repositório    │ ──── push ───▶ │  origin        │
 │ local          │ ◀─── fetch ─── │  (remoto)      │
 └────────────────┘                └────────────────┘
```

| Comando | O que faz |
|---------|-----------|
| `git remote -v` | Lista os remotos configurados |
| `git remote add origin <url>` | Conecta a um remoto |
| `git remote remove origin` | Desconecta |
| `git remote set-url origin <url>` | Troca a URL (ex.: HTTPS → SSH) |

In [ ]:
# Vamos simular um servidor remoto localmente, com um repositório "bare".
# Um repo bare não tem working directory — é só o histórico.
# É exatamente o que o GitHub hospeda.

SERVIDOR = BASE / "servidor_ficticio.git"
subprocess.run(["git", "init", "--bare", "-b", "main", str(SERVIDOR)],
               capture_output=True, text=True)

print("📡 'Servidor' criado em:", SERVIDOR)
print("   Conteúdo (repare: sem arquivos do projeto, só o histórico):")
import os
for item in sorted(os.listdir(SERVIDOR)):
    print("     ", item)

In [ ]:
git("remote", "add", "origin", str(SERVIDOR))
git("remote", "-v")

## 7. `push`, `fetch` e `pull`

| Comando | Direção | O que faz |
|---------|---------|-----------|
| `git push` | local → remoto | Envia seus commits |
| `git fetch` | remoto → local | **Baixa** os commits, **sem** mesclar |
| `git pull` | remoto → local | `fetch` + `merge` |
| `git clone <url>` | remoto → local | Cria uma cópia completa |

> 💡 **`fetch` é seguro, `pull` mexe nos seus arquivos.** Quando estiver em dúvida sobre o que vem por aí, use `git fetch` e depois `git log origin/main --oneline` para inspecionar antes de mesclar.

In [ ]:
# O primeiro push precisa do -u para vincular o branch local ao remoto
git("push", "-u", "origin", "main")

In [ ]:
git("branch", "-vv")     # -vv mostra o vínculo com o remoto

> 📌 **`-u` (ou `--set-upstream`)** cria o vínculo `main → origin/main`. Depois disso, `git push` e `git pull` sem argumentos já sabem para onde ir. Você usa `-u` **uma vez por branch**.

In [ ]:
# Simulando outra pessoa: alguém clona, altera e envia
COLEGA = BASE / "clone_do_colega"
subprocess.run(["git", "clone", str(SERVIDOR), str(COLEGA)], capture_output=True, text=True)

subprocess.run(["git", "config", "user.name", "Colega Aurora"], cwd=COLEGA, capture_output=True)
subprocess.run(["git", "config", "user.email", "colega@aurora.com.br"], cwd=COLEGA, capture_output=True)

escrever("formatacao.py", '''"""Formatação de valores no padrão brasileiro."""


def formatar_brl(valor: float) -> str:
    """Formata como R$ 1.234,56."""
    texto = f"{valor:,.2f}"
    return "R$ " + texto.replace(",", "X").replace(".", ",").replace("X", ".")
''', pasta=COLEGA)

git("add", ".", cwd=COLEGA, mostrar=False)
git("commit", "-m", "feat: adiciona formatação monetária brasileira", cwd=COLEGA, mostrar=False)
git("push", cwd=COLEGA)

In [ ]:
# Do seu lado: fetch baixa, mas NÃO altera seus arquivos
git("fetch")
print(">>> O arquivo do colega já está no meu disco?",
      (REPO / "formatacao.py").exists())
git("log", "--oneline", "--all", "--graph")

In [ ]:
# Inspecionando o que veio, ANTES de mesclar
git("log", "origin/main", "--oneline", "-3")
git("diff", "HEAD", "origin/main", "--stat")

In [ ]:
# Agora sim: pull (= fetch + merge)
git("pull")
print(">>> E agora?", (REPO / "formatacao.py").exists())

### Quando o `push` é rejeitado

```
! [rejected] main -> main (fetch first)
error: failed to push some refs
```

**Tradução:** o remoto tem commits que você não tem. O Git recusa para não apagar o trabalho de outra pessoa.

**Solução correta:**

```bash
git pull          # traz e mescla (pode dar conflito — resolva)
git push          # agora vai
```

> 🔴 **NUNCA resolva isso com `git push --force`.** O `--force` sobrescreve o histórico do servidor e **apaga os commits dos outros**. É a forma mais rápida de estragar o dia de um time inteiro.
>
> Se você realmente precisar forçar (reescreveu o histórico de um branch **seu**, que ninguém mais usa), use `git push --force-with-lease`. Ele recusa se alguém tiver enviado algo desde o seu último fetch — é um `--force` com cinto de segurança.

In [ ]:
# Demonstração da rejeição
escrever("relatorio.py", "# meu trabalho local\n")
git("add", "."); git("commit", "-m", "feat: inicia módulo de relatório", mostrar=False)

# Enquanto isso, o colega também enviou algo
escrever("leitura.py", "# trabalho do colega\n", pasta=COLEGA)
git("add", ".", cwd=COLEGA, mostrar=False)
git("commit", "-m", "feat: inicia módulo de leitura", cwd=COLEGA, mostrar=False)
git("push", cwd=COLEGA, mostrar=False)

print(">>> Tentando push com o remoto à frente:")
git("push")

In [ ]:
# A solução: pull e depois push
git("pull", "--no-rebase", "-m", "merge: sincroniza com origin/main")
git("push")
git("log", "--oneline", "--graph", "-6")

## 8. GitHub: conectando de verdade

### Criando o repositório

1. Entre em [github.com](https://github.com) → botão **New**
2. Nome: `atlas` (ou o que preferir)
3. **Não** marque "Add a README" se você já tem commits locais — isso cria históricos divergentes e complica o primeiro push
4. Copie a URL

### Conectando um projeto local existente

```bash
git remote add origin git@github.com:seu-usuario/atlas.git
git branch -M main
git push -u origin main
```

### Clonando um projeto existente

```bash
git clone git@github.com:seu-usuario/atlas.git
cd atlas
```

### HTTPS vs SSH

| | HTTPS | SSH |
|---|-------|-----|
| URL | `https://github.com/user/repo.git` | `git@github.com:user/repo.git` |
| Autenticação | Personal Access Token | Par de chaves criptográficas |
| Pede senha? | Sim (ou usa credential helper) | Não, depois de configurado |
| Atravessa firewall corporativo | Geralmente sim | Às vezes bloqueado |
| Recomendação | Aceitável | ✅ **Preferível** |

> ⚠️ **A senha da sua conta GitHub não funciona mais no `git push` desde 2021.** Se você usa HTTPS, precisa de um *Personal Access Token* (Settings → Developer settings → Personal access tokens). Por isso SSH é mais confortável: configura uma vez e esquece.

### Configurando SSH — passo a passo

**1. Verifique se já tem uma chave**

```bash
ls -al ~/.ssh
```

Se existir `id_ed25519.pub`, você já tem. Pule para o passo 3.

**2. Gere o par de chaves**

```bash
ssh-keygen -t ed25519 -C "seu@email.com"
```

- Pressione Enter para aceitar o local padrão
- Defina uma senha (*passphrase*) — recomendado
- Isso cria **dois** arquivos:
  - `id_ed25519` → 🔴 **chave privada. NUNCA compartilhe, nunca commite.**
  - `id_ed25519.pub` → 🟢 chave pública. É esta que você entrega ao GitHub.

**3. Copie a chave pública**

```bash
cat ~/.ssh/id_ed25519.pub          # macOS/Linux
type $env:USERPROFILE\.ssh\id_ed25519.pub    # Windows PowerShell
```

**4. Cadastre no GitHub**

Settings → SSH and GPG keys → **New SSH key** → cole → Add.

**5. Teste**

```bash
ssh -T git@github.com
```

Resposta esperada: `Hi seu-usuario! You've successfully authenticated...`

**6. Se já tinha um remoto em HTTPS, troque**

```bash
git remote set-url origin git@github.com:seu-usuario/atlas.git
git remote -v
```

## 9. Pull Requests — o fluxo profissional

Um **Pull Request** (PR) é um pedido: *"revisem estas mudanças e, se aprovarem, mesclem em `main`"*.

É onde acontece a revisão de código, a discussão técnica e a execução automática dos testes.

### O fluxo completo

```bash
# 1. Partindo de main atualizado
git switch main
git pull

# 2. Branch novo
git switch -c feature/relatorio-por-canal

# 3. Trabalha e commita (várias vezes)
git add .
git commit -m "feat(metricas): adiciona agrupamento por canal"

# 4. Envia o branch
git push -u origin feature/relatorio-por-canal

# 5. Abre o PR no GitHub (o próprio push imprime o link)

# 6. Revisão → ajustes → novos commits → push (o PR atualiza sozinho)

# 7. Aprovado → Merge pelo botão do GitHub

# 8. Limpeza local
git switch main
git pull
git branch -d feature/relatorio-por-canal
```

### Um bom PR

- **Título** no padrão de commit: `feat(metricas): adiciona agrupamento por canal`
- **Descrição** respondendo:
  - O que muda?
  - Por quê? (link para o ticket)
  - Como testar?
  - Tem algo que o revisor precisa saber?
- **Pequeno.** Um PR de 200 linhas recebe revisão de verdade. Um de 2.000 recebe "LGTM 👍" e ninguém leu.
- **Autorrevisão primeiro.** Leia o próprio diff no GitHub antes de pedir revisão — você vai achar coisas.

### As três opções de merge no GitHub

| Opção | Resultado | Quando |
|-------|-----------|--------|
| **Create a merge commit** | Preserva todos os commits + commit de merge | Histórico detalhado |
| **Squash and merge** | Todos viram **um** commit em `main` | ✅ Mais comum: `main` fica limpo |
| **Rebase and merge** | Commits reaplicados, sem commit de merge | Histórico linear |

## 🔧 Prática guiada — Fluxo completo com conflito

Vamos simular um dia real: duas pessoas, dois branches, um conflito, um merge.

In [ ]:
import shutil

PRAT = BASE / "pratica"
shutil.rmtree(PRAT, ignore_errors=True)
PRAT.mkdir()

SRV = PRAT / "origin.git"
subprocess.run(["git", "init", "--bare", "-b", "main", str(SRV)], capture_output=True)

DEV_A = PRAT / "dev_ana"
DEV_B = PRAT / "dev_bruno"


def setup(caminho, nome, email):
    subprocess.run(["git", "clone", str(SRV), str(caminho)], capture_output=True)
    subprocess.run(["git", "config", "user.name", nome], cwd=caminho, capture_output=True)
    subprocess.run(["git", "config", "user.email", email], cwd=caminho, capture_output=True)


# Ana cria o projeto
DEV_A.mkdir()
subprocess.run(["git", "init", "-b", "main"], cwd=DEV_A, capture_output=True)
subprocess.run(["git", "config", "user.name", "Ana"], cwd=DEV_A, capture_output=True)
subprocess.run(["git", "config", "user.email", "ana@aurora.com.br"], cwd=DEV_A, capture_output=True)
subprocess.run(["git", "remote", "add", "origin", str(SRV)], cwd=DEV_A, capture_output=True)

escrever("relatorio.py", '''"""Relatório de vendas."""


def gerar(vendas):
    """Monta o relatório em texto."""
    linhas = ["RELATÓRIO DE VENDAS", "=" * 30]
    for v in vendas:
        linhas.append(f"{v['cidade']}: {v['valor']}")
    return "\\n".join(linhas)
''', pasta=DEV_A)

git("add", ".", cwd=DEV_A, mostrar=False)
git("commit", "-m", "feat: relatório inicial", cwd=DEV_A, mostrar=False)
git("push", "-u", "origin", "main", cwd=DEV_A)

# Bruno clona
setup(DEV_B, "Bruno", "bruno@aurora.com.br")
print("✅ Ana e Bruno prontos, ambos partindo do mesmo commit")

In [ ]:
# --- ANA trabalha em um branch ---
git("switch", "-c", "feature/formata-moeda", cwd=DEV_A, mostrar=False)
escrever("relatorio.py", '''"""Relatório de vendas."""


def formatar_brl(valor: float) -> str:
    """Formata como R$ 1.234,56."""
    return "R$ " + f"{valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


def gerar(vendas):
    """Monta o relatório em texto."""
    linhas = ["RELATÓRIO DE VENDAS", "=" * 30]
    for v in vendas:
        linhas.append(f"{v['cidade']}: {formatar_brl(v['valor'])}")
    return "\\n".join(linhas)
''', pasta=DEV_A)

git("add", ".", cwd=DEV_A, mostrar=False)
git("commit", "-m", "feat(relatorio): formata valores em reais", cwd=DEV_A, mostrar=False)
git("push", "-u", "origin", "feature/formata-moeda", cwd=DEV_A)

In [ ]:
# --- BRUNO trabalha na MESMA função, direto em main ---
escrever("relatorio.py", '''"""Relatório de vendas."""


def gerar(vendas):
    """Monta o relatório em texto, ordenado por valor."""
    linhas = ["RELATÓRIO DE VENDAS — AURORA COMÉRCIO", "=" * 40]
    for v in sorted(vendas, key=lambda x: -x["valor"]):
        linhas.append(f"{v['cidade']:<20} {v['valor']:>12}")
    linhas.append("=" * 40)
    return "\\n".join(linhas)
''', pasta=DEV_B)

git("add", ".", cwd=DEV_B, mostrar=False)
git("commit", "-m", "feat(relatorio): ordena por valor e alinha colunas", cwd=DEV_B, mostrar=False)
git("push", cwd=DEV_B)

In [ ]:
# --- ANA tenta integrar: main andou ---
git("switch", "main", cwd=DEV_A, mostrar=False)
git("pull", cwd=DEV_A)
git("merge", "feature/formata-moeda", cwd=DEV_A)

In [ ]:
print(ler("relatorio.py", pasta=DEV_A))

In [ ]:
# Ana resolve COMBINANDO as duas contribuições
escrever("relatorio.py", '''"""Relatório de vendas."""


def formatar_brl(valor: float) -> str:
    """Formata como R$ 1.234,56."""
    return "R$ " + f"{valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


def gerar(vendas):
    """Monta o relatório em texto, ordenado por valor e com moeda formatada."""
    linhas = ["RELATÓRIO DE VENDAS — AURORA COMÉRCIO", "=" * 40]
    for v in sorted(vendas, key=lambda x: -x["valor"]):
        linhas.append(f"{v['cidade']:<20} {formatar_brl(v['valor']):>18}")
    linhas.append("=" * 40)
    return "\\n".join(linhas)
''', pasta=DEV_A)

git("add", "relatorio.py", cwd=DEV_A, mostrar=False)
git("commit", "-m", "merge: combina formatação em reais com ordenação por valor", cwd=DEV_A)
git("push", cwd=DEV_A)
git("log", "--oneline", "--graph", "--all", cwd=DEV_A)

In [ ]:
# O resultado funciona?
teste = '''
import sys
sys.path.insert(0, ".")
from relatorio import gerar

vendas = [
    {"cidade": "Campinas", "valor": 182450.30},
    {"cidade": "São Paulo", "valor": 298100.00},
    {"cidade": "Sorocaba", "valor": 44200.00},
]
print(gerar(vendas))
'''
(DEV_A / "_teste.py").write_text(teste, encoding="utf-8")
r = subprocess.run(["python", "_teste.py"], cwd=DEV_A, capture_output=True, text=True)
print(r.stdout or r.stderr)
(DEV_A / "_teste.py").unlink()

In [ ]:
# Bruno sincroniza e recebe tudo
git("pull", cwd=DEV_B)
print(">>> Bruno tem a função de formatação?",
      "formatar_brl" in ler("relatorio.py", pasta=DEV_B))
git("log", "--oneline", "--graph", "-5", cwd=DEV_B)

## 📝 Exercícios rápidos

**E1.** No laboratório, crie um branch `feature/desconto`, adicione uma função `calcular_desconto`, commite e mescle em `main` com fast-forward. Confirme com `git log --graph` que não houve commit de merge.

**E2.** Crie dois branches a partir de `main`, faça commits em **arquivos diferentes** em cada um, e mescle os dois. Deve funcionar sem conflito. Explique por quê.

**E3.** Provoque um conflito de propósito: dois branches alterando a mesma linha. Resolva de três formas diferentes (ficando com a sua, com a que entra, e combinando). Use `git merge --abort` entre as tentativas.

**E4.** Crie um repositório bare local, adicione como `origin`, faça push, clone em outra pasta, altere no clone, push, e sincronize de volta com pull.

**E5.** Simule a rejeição de push: faça commits divergentes nos dois clones e tente push no que está atrasado. Resolva corretamente.

**E6.** Liste os branches mesclados (`git branch --merged`) e os não mesclados (`git branch --no-merged`). Para que serve cada um no dia a dia?

**E7.** (Real) Crie um repositório no seu GitHub, configure SSH, e envie o `projeto_Atlas` para lá.

In [ ]:
# E1

In [ ]:
# E2

In [ ]:
# E3

In [ ]:
# E4

In [ ]:
# E5

In [ ]:
# E6

## 📋 Cola de referência

```bash
# ── Branches ──
git branch                        # lista locais
git branch -a                     # lista todos (inclui remotos)
git branch -vv                    # com vínculo de upstream
git switch nome                   # troca
git switch -c nome                # cria e troca
git switch -                      # volta ao branch anterior
git branch -d nome                # apaga (seguro)
git branch -D nome                # apaga à força
git branch -m novo-nome           # renomeia o atual
git branch --merged               # já integrados (podem ser apagados)
git branch --no-merged            # ainda não integrados

# ── Merge ──
git merge outro-branch
git merge --no-ff outro-branch    # força commit de merge
git merge --squash outro-branch   # junta tudo num commit
git merge --abort                 # 🆘 cancela e volta

# ── Conflitos ──
git status                        # quais arquivos conflitaram
# ... edite os arquivos, apague os marcadores ...
git add arquivo-resolvido
git commit

# ── Remotos ──
git remote -v
git remote add origin <url>
git remote set-url origin <url>
git remote remove origin

# ── Sincronização ──
git clone <url>
git clone <url> pasta-destino
git fetch                         # baixa SEM mesclar (seguro)
git fetch --prune                 # limpa branches remotos apagados
git pull                          # fetch + merge
git push
git push -u origin nome-do-branch # primeira vez
git push --force-with-lease       # forçar COM segurança
git push -d origin nome           # apaga branch remoto

# ── Inspeção ──
git log --oneline --graph --all
git log origin/main --oneline     # o que tem no remoto
git diff main..feature            # diferença entre branches
```

## ✅ Checklist de saída

- [ ] Explico que um branch é um ponteiro para um commit
- [ ] Sei o que é `HEAD` e como ele se move
- [ ] Uso `git switch -c` para criar e trocar
- [ ] Diferencio merge fast-forward de merge de 3 vias
- [ ] Sei quando faz sentido usar `--no-ff`
- [ ] Leio os marcadores `<<<<<<<`, `=======`, `>>>>>>>` sem pânico
- [ ] Resolvo um conflito e sei que `git merge --abort` existe
- [ ] Conheço o GitHub Flow e por que não se commita em `main`
- [ ] Diferencio `fetch` de `pull`
- [ ] Sei por que `git push --force` é perigoso e o que usar no lugar
- [ ] Configurei (ou sei configurar) uma chave SSH
- [ ] Sei abrir um Pull Request e o que faz um bom PR

---

### ➡️ Próxima aula

**`02_03_Desfazendo_Alteracoes.ipynb`** — `restore`, `revert`, `reset`, `stash` e `reflog`. A aula que transforma "perdi tudo" em "resolvo em 30 segundos".